In [ ]:
# Install dependencies
!pip install -q timm pytorch-msssim lpips scikit-image thop gdown transformers accelerate

import os
import shutil
import subprocess

# 1. Clone your patched repository
!git clone https://github.com/sumire25/RSHazeNet.git /kaggle/working/RSHazeNet
%cd /kaggle/working/RSHazeNet

# 2. Setup RRSHID
RRSHID_ZIP = '/kaggle/working/RRSHID.zip'
RRSHID_BASE = '/kaggle/working/RRSHID'
if not os.path.isdir(RRSHID_BASE):
    !wget -q -O {RRSHID_ZIP} "https://github.com/lwCVer/RRSHID/releases/download/dataset/RRSHID.zip"
    !unzip -q {RRSHID_ZIP} -d {RRSHID_BASE}
    !rm -f {RRSHID_ZIP}

# 3. Virtual Dataset Aggregator (Baseline)
UNIFIED_RRSHID = '/kaggle/working/Unified_RRSHID'

def prepare_unified_layout(source_base, unified_base, levels, splits):
    if os.path.exists(unified_base): shutil.rmtree(unified_base)
    for split in splits:
        hazy_dir = os.path.join(unified_base, split, 'hazy')
        gt_dir = os.path.join(unified_base, split, 'GT')
        os.makedirs(hazy_dir, exist_ok=True)
        os.makedirs(gt_dir, exist_ok=True)
        
        for level in levels:
            raw_split = os.path.join(source_base, level, split)
            if not os.path.isdir(raw_split): continue
                
            raw_hazy = os.path.join(raw_split, 'hazy')
            raw_gt = next((os.path.join(raw_split, c) for c in ['GT', 'gt', 'clear'] 
                           if os.path.isdir(os.path.join(raw_split, c))), None)
            
            if not raw_gt or not os.path.isdir(raw_hazy): continue
            
            for fname in os.listdir(raw_hazy):
                if not fname.lower().endswith(('.png', '.jpg', '.jpeg', '.tif')): continue
                unique_fname = f"{level}_{fname}"
                os.symlink(os.path.join(raw_hazy, fname), os.path.join(hazy_dir, unique_fname))
                os.symlink(os.path.join(raw_gt, fname), os.path.join(gt_dir, unique_fname))

prepare_unified_layout(RRSHID_BASE, UNIFIED_RRSHID, 
                       ['thin_fog', 'moderate_fog', 'thick_fog'], ['train', 'val'])

# 4. Train Orchestrator
def run_train(dataset_name, unified_base, val_split):
    print(f"\n{'='*50}\nTraining Baseline Pipeline: {dataset_name}\n{'='*50}")
    
    save_dir = f'/kaggle/working/weights/{dataset_name}_Baseline'
    os.makedirs(save_dir, exist_ok=True)
    
    train_cmd = [
        'python', 'train.py',
        '--train_input', f'{unified_base}/train/hazy',
        '--train_target', f'{unified_base}/train/GT',
        '--val_input', f'{unified_base}/{val_split}/hazy',
        '--val_target', f'{unified_base}/{val_split}/GT',
        '--epochs', '100',
        '--batch_size_train', '8',
        '--patch_size_train', '256' if dataset_name == "RRSHID" else '512',
        '--patch_size_val', '256' if dataset_name == "RRSHID" else '512',
        '--save_dir', save_dir
    ]
    
    checkpoint_path = f'{save_dir}/model_best.pth'
    if os.path.exists(checkpoint_path):
        print(f"--> Found existing checkpoint at {checkpoint_path}. Skipping/Resuming training!")
        train_cmd += ['--resume', checkpoint_path]
        
    print("-> Training...")
    subprocess.run(train_cmd, check=True)

# Execute RRSHID
run_train("RRSHID", UNIFIED_RRSHID, "val")

# Commented out as SateHaze1k weights were recovered
# UNIFIED_SATEHAZE = '/kaggle/working/Unified_SateHaze1k'
# prepare_unified_layout('/kaggle/input/datasets/xuxingxing233/satehaze1k', UNIFIED_SATEHAZE, ['Haze1k_thin', 'Haze1k_moderate', 'Haze1k_thick'], ['train', 'test'])
# run_train("SateHaze1k", UNIFIED_SATEHAZE, "test")

print("\nRRSHID Training Complete! Output artifacts will be saved.")